In [3]:
# Core imports
import sys
import json
import os
from pathlib import Path

import pandas as pd

_here = Path.cwd()
if (_here / "utils").exists():
    pass
elif (_here / "notebooks" / "utils").exists():
    sys.path.insert(0, str(_here / "notebooks"))

import dspy
from typing import List, Optional, Literal, Union
from pydantic import BaseModel, Field
from tqdm import tqdm


In [4]:
# DSPy Configuration
lm = dspy.LM(
    model="openai/gpt-5.2", 
    api_key=os.environ["OPENAI_API_KEY"],
)
dspy.configure(lm=lm)



### Core Policy Model

In [5]:
from utils.schemas import ExtractedPolicy

In [6]:
from utils.schemas import DocumentMetadata 


---

## 🧠 DSPy Signatures & Modules {#dspy}

Define DSPy signatures and custom modules for policy extraction, validation, and processing.

### Policy Extraction Signature

In [7]:
from utils.dspy_extraction import PolicyExtractionSignature, PolicyExtractor

In [8]:
# (Legacy strict validator removed)
# Individual validation uses the refined validator in `utils.dspy_validation`.
from utils.dspy_validation import PolicyValidationSignature, PolicyValidator, ValidationMetrics

In [9]:
from utils.docling_io import pdf_to_markdown

---

## 🗂️ Batch Configuration

Define the list of cities to process. Add a `DocumentMetadata` + PDF path per city. All 7 steps below will loop over this list automatically.

In [10]:
# ─── Batch Configuration ────────────────────────────────────────────────────
# Each entry has:
#   metadata     -> DocumentMetadata (country, state_or_province, city)
#   pdf_path     -> original PDF (only used if markdown_path is missing)
#   markdown_path -> pre-converted markdown file (skips PDF conversion entirely)
#
# Outputs are written to OUTPUT_DIR/<city_key>/

from utils.schemas import DocumentMetadata

BATCH = [
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Illinois", city="Chicago"),
        "pdf_path": "../pdfs/Chicago-CAP-071822.pdf",
        "markdown_path": "../docs/cities/chicago.md",
    },
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Washington", city="Seattle"),
        "pdf_path": "../pdfs/Seattle.pdf",
        "markdown_path": "../docs/cities/seattle_markdown.md",
    },
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Nevada", city="Las Vegas"),
        "pdf_path": "../pdfs/CLV.pdf",
        "markdown_path": "../docs/cities/LV.md",
    },
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Florida", city="Miami-Dade"),
        "pdf_path": "../pdfs/MIAMI_DADE.pdf",
        "markdown_path": "../docs/cities/miami_markdown.md",
    },
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Texas", city="Austin"),
        "pdf_path": "../pdfs/Austin_climate_equity.pdf",
        "markdown_path": "../docs/cities/austin.md",
    },
    {
        "metadata": DocumentMetadata(country="Senegal", city="Dakar"),
        "pdf_path": "../pdfs/Dakar.pdf",
        "markdown_path": "../docs/cities/dakar.md",
    },
    {
        "metadata": DocumentMetadata(country="Kuwait"),
        "pdf_path": "../pdfs/Kuwait-NAP-2019-2030.pdf",
        "markdown_path": "../docs/cities/kuwait.md",
    },
    {
        "metadata": DocumentMetadata(country="Portugal"),
        "pdf_path": "../pdfs/2021 Portugal ADCOM_UNFCCC.pdf",
        "markdown_path": "../docs/cities/Portugal.md",
    },
    {
        "metadata": DocumentMetadata(country = "Switzerland",city= "Geneva"),
        "pdf_path":"../pdfs/city_of_geneva.pdf",
        "markdown_path":"../docs/cities/geneva.md",

    },
    {
        "metadata":DocumentMetadata(country="Japan",city="Hiroshima"),
        "pdf_path":"../pdfs/HIROSHIMA.pdf",
        "markdown_path":"../docs/cities/Hiroshima.md",
    }
]

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Number of parallel threads for LLM calls (Steps 5 & 6).
NUM_THREADS = 8

def city_key(metadata: DocumentMetadata) -> str:
    """Slug used as filename prefix and output folder name."""
    parts = [metadata.city or metadata.country, metadata.country]
    # e.g. 'Chicago_United_States' or 'Kuwait_Kuwait' → just use city if present
    label = metadata.city if metadata.city else metadata.country
    return label.replace(" ", "_").replace("-", "_")

print(f"Batch configured ({len(BATCH)} cities):")
for e in BATCH:
    key = city_key(e["metadata"])
    md = e.get("markdown_path", "—")
    print(f"  {key:20s}  md={md}")
print(f"\nParallel threads: {NUM_THREADS}")


Batch configured (10 cities):
  Chicago               md=../docs/cities/chicago.md
  Seattle               md=../docs/cities/seattle_markdown.md
  Las_Vegas             md=../docs/cities/LV.md
  Miami_Dade            md=../docs/cities/miami_markdown.md
  Austin                md=../docs/cities/austin.md
  Dakar                 md=../docs/cities/dakar.md
  Kuwait                md=../docs/cities/kuwait.md
  Portugal              md=../docs/cities/Portugal.md
  Geneva                md=../docs/cities/geneva.md
  Hiroshima             md=../docs/cities/Hiroshima.md

Parallel threads: 8


### Step 1 — Load Markdown

Loads each city's document as markdown text using this priority order:
1. **Pre-converted file** supplied via `markdown_path` in `BATCH` (e.g. `docs/cities/*.md`) — instant, no API call
2. **Cached conversion** saved by a previous run (`outputs/{key}_markdown.md`)
3. **Live PDF conversion** via Docling — only runs if neither of the above exists, and caches the result


In [9]:
# Step 1: Load Markdown (pre-converted files take priority; falls back to PDF conversion)
from utils.docling_io import pdf_to_markdown

markdowns = {}  # city_key -> markdown string

for entry in BATCH:
    key = city_key(entry["metadata"])
    md_path = entry.get("markdown_path")

    # ── Priority 1: pre-existing markdown file supplied in config ────────
    if md_path and Path(md_path).exists():
        with open(md_path, "r", encoding="utf-8") as f:
            markdowns[key] = f.read()
        print(f"[{key}] Loaded pre-converted markdown from {md_path}  ({len(markdowns[key]):,} chars)")

    # ── Priority 2: cached conversion saved by a previous run ────────────
    elif (OUTPUT_DIR / f"{key}_markdown.md").exists():
        cached = OUTPUT_DIR / f"{key}_markdown.md"
        with open(cached, "r", encoding="utf-8") as f:
            markdowns[key] = f.read()
        print(f"[{key}] Loaded cached markdown from {cached}  ({len(markdowns[key]):,} chars)")

    # ── Priority 3: convert from PDF and cache the result ────────────────
    else:
        pdf = entry.get("pdf_path")
        if not pdf or not Path(pdf).exists():
            print(f"[{key}]  No markdown and no PDF found — skipping.")
            continue
        cache_path = OUTPUT_DIR / f"{key}_markdown.md"
        print(f"[{key}] Converting PDF: {pdf} ...")
        markdowns[key] = pdf_to_markdown(pdf, save_markdown_path=str(cache_path))
        print(f"[{key}] Done  ({len(markdowns[key]):,} chars)  → cached at {cache_path}")


[Chicago] Loaded pre-converted markdown from ../docs/cities/chicago.md  (256,905 chars)
[Seattle] Loaded pre-converted markdown from ../docs/cities/seattle_markdown.md  (49,888 chars)
[Las_Vegas] Loaded pre-converted markdown from ../docs/cities/LV.md  (61,953 chars)
[Miami_Dade] Loaded pre-converted markdown from ../docs/cities/miami_markdown.md  (141,191 chars)
[Austin] Loaded pre-converted markdown from ../docs/cities/austin.md  (413,340 chars)
[Dakar] Loaded pre-converted markdown from ../docs/cities/dakar.md  (227,420 chars)
[Kuwait] Loaded pre-converted markdown from ../docs/cities/kuwait.md  (362,332 chars)
[Portugal] Loaded pre-converted markdown from ../docs/cities/Portugal.md  (438,819 chars)
[Geneva] Loaded pre-converted markdown from ../docs/cities/geneva.md  (19,174 chars)
[Hiroshima] Loaded pre-converted markdown from ../docs/cities/Hiroshima.md  (87,414 chars)


### Step 2 — Extract Policies

Runs the DSPy `PolicyExtractor` LLM call for each city. Saves `{City_Country}_extracted_policies.json` to `outputs/`.

In [10]:
# Step 2: Extract Policies — one LLM call per city, parallelized across cities
from utils.dspy_extraction import PolicyExtractor
from dspy.utils.parallelizer import ParallelExecutor

policy_extractor = PolicyExtractor()
all_extracted = {}  # city_key -> List[ExtractedPolicy]

# Only extract for cities whose markdown was successfully loaded
batch_to_extract = [e for e in BATCH if city_key(e["metadata"]) in markdowns]

def extract_one(entry):
    key = city_key(entry["metadata"])
    extracted = policy_extractor(
        document_text=markdowns[key],
        document_metadata=entry["metadata"],
    )
    return {"key": key, "extracted": extracted}

print(f"Extracting policies for {len(batch_to_extract)} cities across {NUM_THREADS} threads...\n")

executor = ParallelExecutor(
    num_threads=min(NUM_THREADS, len(batch_to_extract)),
    max_errors=len(batch_to_extract),
    provide_traceback=True,
)
raw_results = executor.execute(extract_one, batch_to_extract)

# ── Collect results in memory, then write — no concurrent file I/O ──────────
n_errors = 0
for i, res in enumerate(raw_results):
    if res is None or isinstance(res, Exception):
        n_errors += 1
        key = city_key(batch_to_extract[i]["metadata"])
        print(f"  ⚠ [{key}] extraction failed: {res}")
    else:
        key = res["key"]
        all_extracted[key] = res["extracted"]

# ── Write all JSON files after all threads are done ──────────────────────────
for key, extracted in all_extracted.items():
    out = OUTPUT_DIR / f"{key}_extracted_policies.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump([p.model_dump() for p in extracted], f, ensure_ascii=False, indent=2)
    print(f"[{key}] {len(extracted)} policies → {out}")

print(f"\nDone. {len(all_extracted)} succeeded, {n_errors} failed.")


Extracting policies for 10 cities across 8 threads...

Processed 10 / 10 examples: 100%|██████████| 10/10 [00:00<00:00, 31.55it/s]
[Chicago] 68 policies → outputs/Chicago_extracted_policies.json
[Seattle] 27 policies → outputs/Seattle_extracted_policies.json
[Las_Vegas] 22 policies → outputs/Las_Vegas_extracted_policies.json
[Miami_Dade] 39 policies → outputs/Miami_Dade_extracted_policies.json
[Austin] 109 policies → outputs/Austin_extracted_policies.json
[Dakar] 17 policies → outputs/Dakar_extracted_policies.json
[Kuwait] 45 policies → outputs/Kuwait_extracted_policies.json
[Portugal] 22 policies → outputs/Portugal_extracted_policies.json
[Geneva] 12 policies → outputs/Geneva_extracted_policies.json
[Hiroshima] 46 policies → outputs/Hiroshima_extracted_policies.json

Done. 10 succeeded, 0 failed.


### Step 3 — Cluster Policies

Groups extracted policies by section header into `parent_with_subs`, `individual`, and `orphan_sub` clusters. Saves `{City_Country}_policy_clusters.json`.

In [11]:
# Step 3: Cluster Policies
from utils.clustering import cluster_policies, summarize_clusters

all_clusters = {}  # city_key -> List[dict]

for entry in BATCH:
    key = city_key(entry["metadata"])
    clusters = cluster_policies(all_extracted[key])
    all_clusters[key] = clusters

    print(f"\n[{key}]")
    summarize_clusters(clusters)

    out = OUTPUT_DIR / f"{key}_policy_clusters.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(clusters, f, default=lambda o: o.model_dump(), ensure_ascii=False, indent=2)
    print(f"  → {out}")



[Chicago]
Total clusters:        17
  Parent+sub clusters: 14
  Individual clusters: 3
  Orphaned subs:       0

[Parent 1] [S T R A T E G Y 1 | RETROFIT BUILDINGS]
  Retrofit buildings to reduce energy use and GHG emissions across residential, mu
  └─ Sub 1: Retrofit residential buildings with 4 or fewer units: 20% by 2030 and 
  └─ Sub 2: Retrofit 20% of total 5+ unit residential buildings by 2030.
  └─ Sub 3: Retrofit 20% of total industrial buildings by 2030.
  └─ Sub 4: Retrofit 90% of total city-owned and sister agency-owned buildings by 
  └─ Sub 5: Retrofit 20% of total commercial buildings by 2035.
[Parent 2] [S T R A T E G Y 2 | CONNECT COMMUNITIES TO RENEWABLE ENERGY]
  Connect communities to renewable energy through community ownership models and e
  └─ Sub 1: Install 5 megawatts of co-owned community solar projects by 2025.
  └─ Sub 2: Increase Chicago-based community renewables to 20 megawatts by 2025.
  └─ Sub 3: Increase community renewables subscriptions so that low-i

### Step 4 — Build Policy Records

Flattens each city's clusters into a structured DataFrame with consistent columns. Saves `{City_Country}_policy_records.csv`.

In [14]:
# Step 4: Build Policy Records DataFrames
from utils.clustering import clusters_to_records

FRONT_COLS = [
    "cluster_id", "cluster_type", "role", "section_header", "sector",
    "policy_statement", "parent_statement", "verbatim_text", "extraction_rationale",
]

all_policy_records = {}  # city_key -> List[dict]
all_df_policies = {}     # city_key -> pd.DataFrame

for entry in BATCH:
    key = city_key(entry["metadata"])
    records = clusters_to_records(all_clusters[key])
    df = pd.DataFrame(records)
    df = df[FRONT_COLS + [c for c in df.columns if c not in FRONT_COLS]]

    all_policy_records[key] = records
    all_df_policies[key] = df

    # out = OUTPUT_DIR / f"{key}_policy_records.csv"
    # df.to_csv(out, index=False)
    # print(f"[{key}] {len(df)} records → {out}")


### Step 5 — Validate Individual Policies

Runs `PolicyValidator` on each `individual`-role policy. Flattens Pydantic metrics into columns. Saves `{City_Country}_validation_individual.csv`.

In [16]:
# Step 5: Validate Individual Policies — parallel LLM calls via DSPy ParallelExecutor
from utils.dspy_validation import PolicyValidator
from dspy.utils.parallelizer import ParallelExecutor

validator = PolicyValidator()
all_df_final = {}  # city_key -> pd.DataFrame (flattened metrics)

for entry in BATCH:
    key = city_key(entry["metadata"])
    individual = [r for r in all_policy_records[key] if r.get("role") == "individual"]
    print(f"\n[{key}] Validating {len(individual)} individual policies "
          f"across {NUM_THREADS} threads...")

    # ── Build per-record task function ──────────────────────────────────
    def validate_one(record):
        pred = validator(policy_data=record)
        return {**record, **pred.toDict()}

    # ── Execute in parallel — results land in-order, errors → None ──────
    executor = ParallelExecutor(
        num_threads=NUM_THREADS,
        max_errors=len(individual),   # don't abort on individual failures
        provide_traceback=True,
    )
    raw_results = executor.execute(validate_one, individual)

    # ── Filter out None (failed calls) and collect errors ────────────────
    validated = []
    n_errors = 0
    for i, res in enumerate(raw_results):
        if res is None or isinstance(res, Exception):
            n_errors += 1
            print(f"  ⚠ Record {i} failed: {res}")
        else:
            validated.append(res)

    print(f"[{key}] {len(validated)} succeeded, {n_errors} failed")

    # ── Build DataFrame (all results now in memory) ───────────────────────
    df_val = pd.DataFrame(validated)

    # Flatten Pydantic validation_results column into separate columns
    if "validation_results" in df_val.columns:
        vdata = df_val["validation_results"].apply(
            lambda x: x.model_dump() if hasattr(x, "model_dump") else x.dict()
        )
        df_metrics = pd.json_normalize(vdata)
        df_val = pd.concat([df_val.drop(columns=["validation_results"]), df_metrics], axis=1)

    all_df_final[key] = df_val

    # ── Write once, after all results are collected ───────────────────────
    # out = OUTPUT_DIR / f"{key}_validation_individual.csv"
    # df_val.to_csv(out, index=False)
    # print(f"[{key}] Saved → {out}")



[Chicago] Validating 3 individual policies across 8 threads...
Processed 3 / 3 examples: 100%|██████████| 3/3 [00:00<00:00, 96.98it/s]
[Chicago] 3 succeeded, 0 failed

[Seattle] Validating 14 individual policies across 8 threads...
Processed 14 / 14 examples: 100%|██████████| 14/14 [00:00<00:00, 121.70it/s]
[Seattle] 14 succeeded, 0 failed

[Las_Vegas] Validating 7 individual policies across 8 threads...
Processed 7 / 7 examples: 100%|██████████| 7/7 [00:00<00:00, 164.34it/s]
[Las_Vegas] 7 succeeded, 0 failed

[Miami_Dade] Validating 12 individual policies across 8 threads...
Processed 12 / 12 examples: 100%|██████████| 12/12 [00:00<00:00, 145.27it/s]
[Miami_Dade] 12 succeeded, 0 failed

[Austin] Validating 13 individual policies across 8 threads...
Processed 13 / 13 examples: 100%|██████████| 13/13 [00:00<00:00, 150.12it/s]
[Austin] 13 succeeded, 0 failed

[Dakar] Validating 3 individual policies across 8 threads...
Processed 3 / 3 examples: 100%|██████████| 3/3 [00:00<00:00, 99.83it

### Step 6 — Validate Initiatives

Runs `run_initiative_validation` on each city's `parent_with_subs` clusters. Saves `{City_Country}_validated_initiatives.csv`. 


In [17]:
# Step 6: Validate Initiatives — parallel LLM calls via DSPy ParallelExecutor
from dspy.utils.parallelizer import ParallelExecutor
from utils.initiative_validator import InitiativeValidator, build_initiative_context

init_validator = InitiativeValidator()
all_df_initiatives = {}  # city_key -> pd.DataFrame

for entry in BATCH:
    key = city_key(entry["metadata"])
    initiative_clusters = [
        c for c in all_clusters[key] if c["cluster_type"] == "parent_with_subs"
    ]
    if not initiative_clusters:
        print(f"No parent_with_subs clusters — skipping.")
        all_df_initiatives[key] = pd.DataFrame()
        continue

    # Per-cluster task 
    def validate_initiative(cluster):
        context = build_initiative_context(cluster)
        prediction = init_validator(initiative_data=context)
        metrics = prediction.initiative_metrics
        return {
            "initiative_name":              context["initiative_name"],
            "parent_statement":             context["parent_statement"],
            "sector":                       context["sector"],
            "num_sub_actions":              context["num_sub_actions"],
            "coverage_score":               metrics.coverage_score,
            "coverage_assessment":          metrics.coverage_assessment,
            "coherence_score":              metrics.coherence_score,
            "coherence_assessment":         metrics.coherence_assessment,
            "aggregate_measurability":      metrics.aggregate_measurability,
            "has_implementation_pathway":   metrics.has_implementation_pathway,
            "inherited_binding_mechanism":  metrics.inherited_binding_mechanism,
            "inherited_spatial_scope":      metrics.inherited_spatial_scope,
            "weak_signals":                 metrics.weak_signals,
            "strong_signals":               metrics.strong_signals,
            "initiative_result":            metrics.initiative_result,
            "confidence_score":             metrics.confidence_score,
            "initiative_reasoning":         metrics.initiative_reasoning,
            "final_verdict":                metrics.final_verdict,
            "sub_assessments": [
                sa.model_dump() if hasattr(sa, "model_dump") else sa.dict()
                for sa in metrics.sub_assessments
            ],
            "subs_strong":   sum(1 for sa in metrics.sub_assessments if sa.strength == "strong"),
            "subs_moderate": sum(1 for sa in metrics.sub_assessments if sa.strength == "moderate"),
            "subs_weak":     sum(1 for sa in metrics.sub_assessments if sa.strength == "weak"),
        }

    # ── Execute in parallel ───────────────────────────────────────────────
    executor = ParallelExecutor(
        num_threads=NUM_THREADS,
        max_errors=len(initiative_clusters),
        provide_traceback=True,
    )
    raw_results = executor.execute(validate_initiative, initiative_clusters)

    # ── Filter failures ───────────────────────────────────────────────────
    results, n_errors = [], 0
    for i, res in enumerate(raw_results):
        if res is None or isinstance(res, Exception):
            n_errors += 1
            print(f"Cluster {i} failed: {res}")
        else:
            results.append(res)

    print(f"[{key}] {len(results)} succeeded, {n_errors} failed")

    # ── Build DataFrame + print summary ──────────────────────────────────
    df_inits = pd.DataFrame(results)
    if not df_inits.empty:
        print(f"  SOUND:   {(df_inits['initiative_result'] == 'SOUND').sum()}")
        print(f"  PARTIAL: {(df_inits['initiative_result'] == 'PARTIAL').sum()}")
        print(f"  WEAK:    {(df_inits['initiative_result'] == 'WEAK').sum()}")
        print(f"  final_verdict=True: {df_inits['final_verdict'].sum()}")

    all_df_initiatives[key] = df_inits

    #Write once, after all results are collected. initiatives are built 
    # out = OUTPUT_DIR / f"{key}_validation_initiatives.csv"
    # df_inits.to_csv(out, index=False)
    # print(f"[{key}] Saved → {out}")


Processed 14 / 14 examples: 100%|██████████| 14/14 [00:00<00:00, 98.30it/s]
[Chicago] 14 succeeded, 0 failed
  SOUND:   8
  PARTIAL: 5
  WEAK:    1
  final_verdict=True: 13
Processed 1 / 1 examples: 100%|██████████| 1/1 [00:00<00:00, 67.76it/s]
[Seattle] 1 succeeded, 0 failed
  SOUND:   0
  PARTIAL: 0
  WEAK:    1
  final_verdict=True: 0
Processed 4 / 4 examples: 100%|██████████| 4/4 [00:00<00:00, 157.11it/s]
[Las_Vegas] 4 succeeded, 0 failed
  SOUND:   0
  PARTIAL: 3
  WEAK:    1
  final_verdict=True: 3
Processed 7 / 7 examples: 100%|██████████| 7/7 [00:00<00:00, 103.21it/s]
[Miami_Dade] 7 succeeded, 0 failed
  SOUND:   6
  PARTIAL: 1
  WEAK:    0
  final_verdict=True: 7
Processed 18 / 18 examples: 100%|██████████| 18/18 [00:00<00:00, 88.90it/s]
[Austin] 18 succeeded, 0 failed
  SOUND:   3
  PARTIAL: 14
  WEAK:    1
  final_verdict=True: 17
Processed 1 / 1 examples: 100%|██████████| 1/1 [00:00<00:00, 64.93it/s]
[Dakar] 1 succeeded, 0 failed
  SOUND:   0
  PARTIAL: 0
  WEAK:    1
  fin

### Step 7 — Export Combined Results

Builds the final combined table per city and writes `combined_policies.csv`, `trace_individual_policies.csv`, and `trace_initiative_policies.csv` under `outputs/{City_Country}/`.

In [18]:
# Step 7: Export Combined Results per City
from utils.exports import build_combined_policies_table, export_combined_table_and_traces
from utils.exports import filter_valid_policies

all_combined = {}  # city_key -> pd.DataFrame (all policies)
all_valid    = {}  # city_key -> pd.DataFrame (final_verdict == True only)

for entry in BATCH:
    key = city_key(entry["metadata"])
    city_output_dir = OUTPUT_DIR / key
    df_inits = all_df_initiatives.get(key, pd.DataFrame())
    df_inits_arg = df_inits if not df_inits.empty else None

    combined = build_combined_policies_table(
        df_policies=all_df_policies[key],
        df_final_individual=all_df_final[key],
        df_initiatives=df_inits,
        policy_clusters=all_clusters[key],
    )
    all_combined[key] = combined

    # combined_policies.csv now contains valid policies only (individual + initiative clusters)
    # traces contain everything (all rows, full validation detail)
    paths = export_combined_table_and_traces(
        combined=combined,
        df_initiatives=df_inits_arg,
        df_final_individual=all_df_final[key],
        output_dir=city_output_dir,
    )

    # Keep an in-memory copy of valid policies (final_verdict == True)
    df_valid = filter_valid_policies(combined, all_df_final[key], df_inits_arg)
    all_valid[key] = df_valid

    n_all   = len(combined)
    n_valid = len(df_valid)
    print(f"\n[{key}] {n_valid}/{n_all} valid policies  →  {city_output_dir}/combined_policies.csv")
    for k, v in paths.items():
        print(f"  {k}: {v}")


  [build_combined] backfilled 3 individual policies missing from clusters.

[Chicago] 64/68 valid policies  →  outputs/Chicago/combined_policies.csv
  combined_policies: outputs/Chicago/combined_policies.csv
  trace_individual: outputs/Chicago/trace_individual_policies.csv
  trace_individual_valid: outputs/Chicago/trace_individual_policies_valid.csv
  trace_initiatives: outputs/Chicago/trace_initiative_policies.csv
  trace_initiatives_valid: outputs/Chicago/trace_initiative_policies_valid.csv
  [build_combined] backfilled 14 individual policies missing from clusters.

[Seattle] 5/15 valid policies  →  outputs/Seattle/combined_policies.csv
  combined_policies: outputs/Seattle/combined_policies.csv
  trace_individual: outputs/Seattle/trace_individual_policies.csv
  trace_individual_valid: outputs/Seattle/trace_individual_policies_valid.csv
  trace_initiatives: outputs/Seattle/trace_initiative_policies.csv
  trace_initiatives_valid: outputs/Seattle/trace_initiative_policies_valid.csv
  [b

In [ ]:
#

In [12]:

# ── Load all_valid from disk for Step 8 (skips Steps 1–7) ────────────────────
#
# CLASSIFICATION_INPUT controls what gets fed into Step 8:
#
#   "valid_only"    — reads combined_policies.csv (individual + parent + sub rows
#                     where final_verdict == True).  Fast; uses the saved output.
#                     ⚠ Cities with no passing initiatives will only have their
#                     individually-validated policies (e.g. Seattle = 5 rows).
#
#   "all_extracted" — re-clusters the cached extracted_policies.json (no LLM calls),
#                     then keeps ALL individual + sub policies regardless of verdict.
#                     Use this when you want to classify every extracted policy,
#                     including sub-actions from initiatives that failed validation.
#
CLASSIFICATION_INPUT = "valid_only"   # "valid_only" | "all_extracted"

# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("outputs")

try:
    BATCH
except NameError:
    from utils.schemas import DocumentMetadata
    BATCH = [
        {"metadata": DocumentMetadata(country="United States", state_or_province="Illinois",  city="Chicago")},
        {"metadata": DocumentMetadata(country="United States", state_or_province="Washington", city="Seattle")},
        {"metadata": DocumentMetadata(country="United States", state_or_province="Nevada",     city="Las Vegas")},
        {"metadata": DocumentMetadata(country="United States", state_or_province="Florida",    city="Miami-Dade")},
        {"metadata": DocumentMetadata(country="United States", state_or_province="Texas",      city="Austin")},
        {"metadata": DocumentMetadata(country="Senegal",                                       city="Dakar")},
        {"metadata": DocumentMetadata(country="Kuwait")},
        {"metadata": DocumentMetadata(country="Portugal")},
        {"metadata": DocumentMetadata(country="Switzerland",                                   city="Geneva")},
        {"metadata": DocumentMetadata(country="Japan",                                         city="Hiroshima")},
    ]

    def city_key(metadata):
        label = metadata.city if metadata.city else metadata.country
        return label.replace(" ", "_").replace("-", "_")

all_valid = {}

if CLASSIFICATION_INPUT == "valid_only":
    # ── Mode 1: read combined_policies.csv (valid rows only) ─────────────────
    for entry in BATCH:
        key = city_key(entry["metadata"])
        csv_path = OUTPUT_DIR / key / "combined_policies.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            all_valid[key] = df
            role_dist = df["role"].value_counts().to_dict() if "role" in df.columns else {}
            print(f"[{key}] {len(df)} valid policies — {role_dist}")
        else:
            print(f"[{key}] ⚠ {csv_path} not found — skipping.")

elif CLASSIFICATION_INPUT == "all_extracted":
    # ── Mode 2: include ALL individual + sub rows without LLM calls ────────────
    # Preferred source: cached extracted JSON (lets us re-run clustering).
    # Fallback source: existing CSV outputs under outputs/{City}/.
    from utils.clustering import cluster_policies, clusters_to_records
    from utils.schemas import ExtractedPolicy
    import ast

    def _ensure_cols(df: pd.DataFrame) -> pd.DataFrame:
        for col in (
            "verbatim_text", "policy_statement", "sector", "section_header",
            "parent_statement", "cluster_id", "cluster_type", "role",
        ):
            if col not in df.columns:
                df[col] = None
        return df

    for entry in BATCH:
        key = city_key(entry["metadata"])

        # 1) Best: cached extracted JSON (flat)
        candidate_jsons = [
            OUTPUT_DIR / f"{key}_extracted_policies.json",
            OUTPUT_DIR / key / "extracted_policies.json",
            OUTPUT_DIR / key / f"{key}_extracted_policies.json",
        ]
        json_path = next((p for p in candidate_jsons if p.exists()), None)

        if json_path is not None:
            with open(json_path, "r", encoding="utf-8") as f:
                raw = json.load(f)

            extracted = [ExtractedPolicy(**p) for p in raw]
            clusters  = cluster_policies(extracted)
            records   = clusters_to_records(clusters)

            df = pd.DataFrame([
                r for r in records
                if r.get("role") in ("individual", "sub", "orphan_sub")
            ])
            if df.empty:
                print(f"[{key}] ⚠ No individual/sub policies found after clustering from {json_path}.")
                continue

            df = _ensure_cols(df)
            all_valid[key] = df
            role_dist = df["role"].value_counts().to_dict()
            print(f"[{key}] {len(df)} policies (all extracted) — {role_dist}  [source: {json_path}]")
            continue

        # 2) Fallback: build a classification input from existing CSV outputs
        city_dir = OUTPUT_DIR / key
        combined_path = city_dir / "combined_policies.csv"
        ind_trace_path = city_dir / "trace_individual_policies.csv"
        init_trace_path = city_dir / "trace_initiative_policies.csv"

        parts = []

        # - subs + some individual rows (usually VALID only)
        if combined_path.exists():
            df_c = pd.read_csv(combined_path)
            if "role" not in df_c.columns:
                df_c["role"] = None
            df_c = df_c[df_c["role"].isin(["individual", "sub", "orphan_sub"])].copy()
            parts.append(df_c)

        # - individual trace (includes VALID + INVALID individuals)
        if ind_trace_path.exists():
            df_i = pd.read_csv(ind_trace_path)
            if "role" not in df_i.columns:
                df_i["role"] = "individual"
            df_i = df_i[df_i["role"].isin(["individual"])].copy()
            parts.append(df_i)

        # - initiative trace: expand sub_assessments into synthetic sub rows (best-effort)
        if init_trace_path.exists():
            df_init = pd.read_csv(init_trace_path)
            synth = []
            for _, row in df_init.iterrows():
                raw_subs = row.get("sub_assessments")
                if not isinstance(raw_subs, str) or not raw_subs.strip():
                    continue
                try:
                    subs = ast.literal_eval(raw_subs)
                except Exception:
                    continue
                if not isinstance(subs, list):
                    continue
                for s in subs:
                    if not isinstance(s, dict):
                        continue
                    stmt = s.get("action_label") or s.get("action")
                    if not stmt:
                        continue
                    synth.append({
                        "role": "sub",
                        "cluster_type": "parent_with_subs",
                        "cluster_id": None,
                        "sector": row.get("sector"),
                        "section_header": row.get("initiative_name"),
                        "parent_statement": row.get("parent_statement"),
                        "policy_statement": stmt,
                        "verbatim_text": stmt,
                    })

            if synth:
                parts.append(pd.DataFrame(synth))

        if not parts:
            print(f"[{key}] ⚠ No cached JSON and no usable CSV outputs under {city_dir} — skipping.")
            continue

        df = pd.concat(parts, ignore_index=True, sort=False)
        df = _ensure_cols(df)

        # keep only rows we can classify
        df = df[df["role"].isin(["individual", "sub", "orphan_sub"])].copy()
        df = df[df["policy_statement"].notna() & (df["policy_statement"].astype(str).str.strip() != "")]

        if df.empty:
            print(f"[{key}] ⚠ Fallback CSV sources produced 0 classify-able rows — skipping.")
            continue

        all_valid[key] = df
        role_dist = df["role"].value_counts().to_dict()
        print(f"[{key}] {len(df)} policies (all extracted, fallback) — {role_dist}  [source: {city_dir}]")

else:
    raise ValueError(f"Unknown CLASSIFICATION_INPUT: {CLASSIFICATION_INPUT!r}. "
                     "Use 'valid_only' or 'all_extracted'.")

total = sum(len(v) for v in all_valid.values())
print(f"\nall_valid populated: {len(all_valid)} cities, {total} total policies  "
      f"(mode: {CLASSIFICATION_INPUT})")


[Chicago] 64 valid policies — {'sub': 49, 'parent': 13, 'individual': 2}
[Seattle] 5 valid policies — {'individual': 5}
[Las_Vegas] 13 valid policies — {'sub': 9, 'parent': 3, 'individual': 1}
[Miami_Dade] 28 valid policies — {'sub': 20, 'parent': 7, 'individual': 1}
[Austin] 93 valid policies — {'sub': 73, 'parent': 17, 'individual': 3}
[Dakar] 3 valid policies — {'individual': 3}
[Kuwait] 14 valid policies — {'sub': 10, 'parent': 3, 'individual': 1}
[Portugal] 2 valid policies — {'individual': 2}
[Geneva] 1 valid policies — {'individual': 1}
[Hiroshima] 3 valid policies — {'individual': 3}

all_valid populated: 10 cities, 226 total policies  (mode: valid_only)


### Step 8 — 3-Stage Registry Classification

Classifies all valid policies (individual + sub rows from `all_valid`) using a **three-stage mechanism registry**:

1. **Stage 1 — Mechanism Extraction** (parallelized, 1 call/policy): strips city/numbers/dates, extracts a canonical `<action> → <climate_effect>` string per policy.  
2. **Stage 2 — Mechanism Classification** (sequential, 1 call per *unique* mechanism): classifies the abstract mechanism *once* with up to 3 cross-city representatives. Labels propagate to all policies sharing that mechanism — **consistency is structural**.  
3. **Stage 3 — Policy Enrichment** (parallelized, 1 call/policy): adds instance-specific fields (`instrument_type`, `instrument_directness`, `climate_relevance`, location-based `additional_secondary`). Stage 2 labels are **locked** — Stage 3 can only add, never change them.

Fields added per policy: `canonical_mechanism`, `mechanism_description`, `primary_category`, `secondary_categories`,  
`secondary_justification`, `primary_causal_pathway`, `causal_mechanism_detail`, `dominant_pathway_test`,  
`mechanism_classification_reasoning`, `mechanism_confidence`, `instrument_type`, `instrument_directness`,  
`climate_relevance`, `additional_secondary`, `additional_secondary_evidence`, `key_indicators`, `co_benefits`.

Saves `classified_policies.csv` and `trace_classification.json` under `outputs/{City}/`.

> **Vulnerability context**: populate `LOCATION_VULNERABILITIES` in `utils/consistent_classification.py` to enable location-based Adaptation secondaries in Stage 3.

In [14]:

# Step 8: Optimized 3-Stage Registry Classification

from utils.consistent_classification import (
    ConsistentPolicyClassifier,
    build_vulnerability_context,
)
from dspy.utils.parallelizer import ParallelExecutor
from collections import defaultdict as _defdict

classifier = ConsistentPolicyClassifier()
all_classified = {}

# ── Pool valid policies from all cities ──────────────────────────────────────
all_records = []
skipped_keys = []
_vuln_cache = {}

for entry in BATCH:
    key = city_key(entry["metadata"])
    df_valid = all_valid.get(key)
    if df_valid is None or df_valid.empty:
        print(f"[{key}] No valid policies — skipping classification.")
        all_classified[key] = pd.DataFrame()
        skipped_keys.append(key)
        continue

    if key not in _vuln_cache:
        _vuln_cache[key] = build_vulnerability_context(key)

    vuln_ctx = _vuln_cache[key]
    for rec in df_valid.to_dict(orient="records"):
        rec["__city_key"] = key
        rec["location"] = key
        rec["location_vulnerability_context"] = vuln_ctx
        all_records.append(rec)

active_keys = [city_key(e["metadata"]) for e in BATCH if city_key(e["metadata"]) not in skipped_keys]
print(f"\nPooled {len(all_records)} valid policies across {len(active_keys)} cities for classification.")

# ── Per-city input breakdown ──────────────────────────────────────────────────
print(f"\n{'─'*60}")
print("INPUT BREAKDOWN BY CITY")
print(f"{'─'*60}")
_by_city = _defdict(list)
for rec in all_records:
    _by_city[rec["__city_key"]].append(rec)
for _k, _recs in _by_city.items():
    _roles = _defdict(int)
    for r in _recs:
        _roles[r.get("role", "unknown")] += 1
    _role_str = ", ".join(f"{v} {k}" for k, v in sorted(_roles.items()))
    print(f"  {_k:<20s} {len(_recs):>4d} policies  [{_role_str}]")
    for r in _recs:
        stmt = str(r.get("policy_statement", ""))[:90]
        print(f"    [{r.get('role','?'):10s}] {stmt}{'…' if len(str(r.get('policy_statement',''))) > 90 else ''}")

if not all_records:
    print("Nothing to classify.")
else:
    _N = max(NUM_THREADS, min(32, len(all_records)))

    # ── STAGE 1: Mechanism Extraction (parallelized) ──────────────────────────
    print(f"\n{'='*60}")
    print(f"STAGE 1: Extracting mechanisms from {len(all_records)} policies ({_N} threads) ...")
    print(f"{'='*60}")

    def _extract_one(record):
        extraction = classifier.extract_mechanism(
            policy_statement=record["policy_statement"],
            verbatim_text=record["verbatim_text"],
        )
        return {
            **record,
            "canonical_mechanism":   extraction.canonical_mechanism,
            "sector":                extraction.sector,
            "mechanism_description": extraction.mechanism_description,
        }

    _exec1 = ParallelExecutor(
        num_threads=_N,
        max_errors=len(all_records),
        provide_traceback=True,
    )
    _s1_raw = _exec1.execute(_extract_one, all_records)

    s1_records, s1_fail = [], 0
    for i, res in enumerate(_s1_raw):
        if res is None or isinstance(res, Exception):
            s1_fail += 1
            print(f"  ⚠ Stage 1 record {i} failed: {res}")
            fallback = dict(all_records[i])
            fallback["canonical_mechanism"]   = "unknown_mechanism → unknown_effect"
            fallback["sector"]                = "cross_sector"
            fallback["mechanism_description"] = "mechanism extraction failed"
            s1_records.append(fallback)
        else:
            s1_records.append(res)

    print(f"Stage 1 complete: {len(s1_records) - s1_fail} succeeded, {s1_fail} failed")

    # ── Stage 1 results: show what each policy mapped to ─────────────────────
    print(f"\n{'─'*60}")
    print("STAGE 1 RESULTS — policy → canonical_mechanism")
    print(f"{'─'*60}")
    for rec in s1_records:
        stmt  = str(rec.get("policy_statement", ""))[:70]
        mech  = rec.get("canonical_mechanism", "—")
        city  = rec.get("__city_key", "?")
        role  = rec.get("role", "?")
        print(f"  [{city:<15s} | {role:<10s}] {stmt:<72s}")
        print(f"    → {mech}")

    # ── STAGE 2: Mechanism Classification (parallelized by unique mechanism) ──
    print(f"\n{'='*60}")
    print(f"STAGE 2: Classifying mechanisms ...")
    print(f"{'='*60}")

    _mech_groups = _defdict(list)
    for rec in s1_records:
        _mech_groups[rec["canonical_mechanism"]].append(rec)

    unique_mechs = list(_mech_groups.keys())
    print(f"  {len(unique_mechs)} unique mechanisms from {len(s1_records)} policies "
          f"({len(s1_records) - len(unique_mechs)} LLM calls saved vs row-by-row)")

    # ── Mechanism groups: show which policies share each mechanism ────────────
    print(f"\n{'─'*60}")
    print("STAGE 2 GROUPS — mechanism → policies that share it")
    print(f"{'─'*60}")
    for mech, recs in sorted(_mech_groups.items(), key=lambda x: -len(x[1])):
        cities = list({r["__city_key"] for r in recs})
        print(f"\n  [{len(recs):>3d} policies | {', '.join(cities)}]  {mech}")
        for r in recs:
            stmt = str(r.get("policy_statement", ""))[:80]
            print(f"      · {stmt}{'…' if len(str(r.get('policy_statement',''))) > 80 else ''}")

    def _classify_mechanism(mech_key):
        reps = _mech_groups[mech_key][:3]
        return mech_key, classifier.classify_mechanism(
            canonical_mechanism=mech_key,
            representative_policies=reps,
        )

    _exec2 = ParallelExecutor(
        num_threads=min(_N, len(unique_mechs)),
        max_errors=len(unique_mechs),
        provide_traceback=True,
    )

    def _safe_get(obj, key, default="—"):
        try:
            return obj[key]
        except Exception:
            if hasattr(obj, "get"):
                try:
                    return obj.get(key, default)
                except Exception:
                    return default
            return default

    def _augment_labels(mech_key, labels):
        recs = _mech_groups.get(mech_key, [])
        cities = sorted({r.get("__city_key", "?") for r in recs})

        base = {}
        if isinstance(labels, dict):
            base = labels
        elif hasattr(labels, "keys"):
            for k in labels.keys():
                try:
                    base[k] = labels[k]
                except Exception:
                    pass

        base = dict(base)
        base.setdefault("policy_count", len(recs))
        base.setdefault("locations", ", ".join(cities))
        return base

    try:
        _s2_raw = _exec2.execute(_classify_mechanism, unique_mechs)
        registry = {}
        for res in _s2_raw:
            if res is None or isinstance(res, Exception):
                print(f"  ⚠ Stage 2 mechanism failed: {res}")
            else:
                mech_key, labels = res
                registry[mech_key] = _augment_labels(mech_key, labels)
        classifier.mechanism_registry = registry
    except (AttributeError, TypeError):
        registry = classifier.stage2_classify_mechanisms(s1_records)
        registry = {k: _augment_labels(k, v) for k, v in registry.items()}
        classifier.mechanism_registry = registry

    print(f"\nMechanism registry: {len(registry)} unique mechanisms")
    for mech, info in registry.items():
        print(f"\n  {mech}")
        print(f"    Primary:   {_safe_get(info, 'primary_category', '—')}")
        print(f"    Secondary: {_safe_get(info, 'secondary_categories', '—')}")
        print(f"    Policies:  {_safe_get(info, 'policy_count', '?')} across {_safe_get(info, 'locations', '?')}")

    # ── Persist mechanism registry with representative policies ─────────────────
    def _repr_policy(r):
        return {
            "policy_statement": str(r.get("policy_statement", ""))[:500],
            "verbatim_text": str(r.get("verbatim_text", ""))[:500],
            "city": r.get("__city_key", "?"),
            "role": r.get("role", "?"),
        }
    def _json_val(v):
        if v is None or isinstance(v, (str, int, float, bool)):
            return v
        return str(v)
    registry_trace = []
    for mech_key, info in registry.items():
        reps = _mech_groups.get(mech_key, [])[:3]
        entry = {"canonical_mechanism": mech_key}
        for k, v in info.items():
            try:
                entry[k] = _json_val(v)
            except Exception:
                entry[k] = str(v) if v is not None else None
        entry["representative_policies"] = [_repr_policy(r) for r in reps]
        entry["all_policy_statements"] = [str(r.get("policy_statement", ""))[:200] for r in _mech_groups.get(mech_key, [])]
        registry_trace.append(entry)
    registry_path = OUTPUT_DIR / "mechanism_registry.json"
    with open(registry_path, "w", encoding="utf-8") as f:
        json.dump(registry_trace, f, ensure_ascii=False, indent=2)
    print(f"\n  → {registry_path}")

    # ── STAGE 3: Policy Enrichment (parallelized) ─────────────────────────────
    print(f"\n{'='*60}")
    print(f"STAGE 3: Enriching {len(s1_records)} policies ({_N} threads) ...")
    print(f"{'='*60}")

    _registry_snapshot = dict(classifier.mechanism_registry)

    def _enrich_one(record):
        mech_key    = record["canonical_mechanism"]
        mech_labels = _registry_snapshot[mech_key]

        enrichment = classifier.enrich_policy(
            policy_statement=record["policy_statement"],
            verbatim_text=record["verbatim_text"],
            assigned_primary=mech_labels["primary_category"],
            assigned_secondaries=mech_labels["secondary_categories"],
            assigned_causal_pathway=mech_labels["primary_causal_pathway"],
            assigned_causal_detail=mech_labels["causal_mechanism_detail"],
            assigned_typical_instruments=mech_labels["typical_policy_instruments"],
            location_vulnerability_context=record.get(
                "location_vulnerability_context", "No vulnerability context provided"
            ),
        )

        result = {
            **record,
            "primary_category":                   mech_labels["primary_category"],
            "secondary_categories":               mech_labels["secondary_categories"],
            "secondary_justification":            mech_labels["secondary_justification"],
            "primary_causal_pathway":             mech_labels["primary_causal_pathway"],
            "causal_mechanism_detail":            mech_labels["causal_mechanism_detail"],
            "dominant_pathway_test":              mech_labels["dominant_pathway_test"],
            "mechanism_classification_reasoning": mech_labels["classification_reasoning"],
            "mechanism_confidence":               mech_labels["confidence_score"],
            "mechanism_edge_case_notes":          mech_labels["edge_case_notes"],
        }

        add_sec = enrichment.additional_secondary
        if add_sec and add_sec.strip().lower() != "none":
            existing = mech_labels["secondary_categories"]
            result["secondary_categories"] = (
                add_sec if not existing or existing.strip().lower() == "none"
                else f"{existing}; {add_sec}"
            )
            result["additional_secondary_evidence"] = enrichment.additional_secondary_evidence
        else:
            result["additional_secondary_evidence"] = "None"

        result["instrument_type"]          = enrichment.instrument_type
        result["instrument_directness"]    = enrichment.instrument_directness
        result["climate_relevance"]        = enrichment.climate_relevance
        result["key_indicators"]           = enrichment.key_indicators
        result["co_benefits"]              = enrichment.co_benefits
        result["instance_edge_case_notes"] = enrichment.edge_case_notes

        return result

    _exec3 = ParallelExecutor(
        num_threads=_N,
        max_errors=len(s1_records),
        provide_traceback=True,
    )
    _s3_raw = _exec3.execute(_enrich_one, s1_records)

    classified_flat, s3_fail = [], 0
    for i, res in enumerate(_s3_raw):
        if res is None or isinstance(res, Exception):
            s3_fail += 1
            print(f"  ⚠ Stage 3 record {i} failed: {res}")
            classified_flat.append({
                **s1_records[i],
                "instrument_type":   "ERROR",
                "climate_relevance": "ERROR",
            })
        else:
            classified_flat.append(res)

    print(f"\nStage 3 complete: {len(classified_flat) - s3_fail} succeeded, {s3_fail} failed")

    # ── Split results back per city and save ──────────────────────────────────
    _per_city = _defdict(list)
    for rec in classified_flat:
        _per_city[rec["__city_key"]].append(rec)

    _TRACE_COLS = [
        "policy_statement", "role",
        "canonical_mechanism", "mechanism_description", "sector",
        "primary_category", "secondary_categories", "secondary_justification",
        "primary_causal_pathway", "causal_mechanism_detail", "dominant_pathway_test",
        "mechanism_classification_reasoning", "mechanism_confidence", "mechanism_edge_case_notes",
        "additional_secondary", "additional_secondary_evidence",
        "instrument_type", "instrument_directness", "climate_relevance",
        "key_indicators", "co_benefits", "instance_edge_case_notes",
    ]

    dfs_to_write = {}
    for entry in BATCH:
        key = city_key(entry["metadata"])
        if key not in _per_city:
            continue
        df_classified = pd.DataFrame(_per_city[key])
        df_classified.drop(columns=["__city_key"], errors="ignore", inplace=True)
        all_classified[key] = df_classified
        dfs_to_write[key] = df_classified

    for key, df_classified in dfs_to_write.items():
        city_output_dir = OUTPUT_DIR / key
        city_output_dir.mkdir(parents=True, exist_ok=True)

        out_csv = city_output_dir / "classified_policies.csv"
        df_classified.to_csv(out_csv, index=False)

        trace_cols = [c for c in _TRACE_COLS if c in df_classified.columns]
        trace_data = df_classified[trace_cols].to_dict(orient="records")
        out_traces = city_output_dir / "trace_classification.json"
        with open(out_traces, "w", encoding="utf-8") as f:
            json.dump(trace_data, f, ensure_ascii=False, indent=2)

        cat_counts = df_classified["primary_category"].value_counts()
        _conf_col  = df_classified.get("mechanism_confidence", pd.Series(dtype=float))
        avg_conf   = _conf_col.mean()
        n_multi    = df_classified["secondary_categories"].apply(
            lambda x: bool(x) and str(x).strip().lower() not in ("none", "")
        ).sum()

        print(f"\n[{key}] ── Classification Summary ──")
        print(f"  Total classified : {len(df_classified)}")
        print(f"  Multi-label      : {n_multi}")
        if pd.notna(avg_conf):
            print(f"  Avg confidence   : {avg_conf:.3f}")
        print(f"  Category distribution:")
        for cat, cnt in cat_counts.items():
            pct = cnt / len(df_classified) * 100
            print(f"    {cat:<30s} {cnt:>4d}  ({pct:.1f}%)")
        print(f"  → {out_csv}")
        print(f"  → {out_traces}")

    print(f"\n{'='*60}")
    print(f"REGISTRY SUMMARY")
    print(f"  Total policies classified : {len(all_records)}")
    print(f"  Unique mechanisms found   : {len(registry)}")
    print(f"  Stage 2 LLM calls saved   : {len(all_records) - len(registry)} "
          f"(vs. row-by-row baseline)")
    print(f"{'='*60}")



Pooled 226 valid policies across 10 cities for classification.

────────────────────────────────────────────────────────────
INPUT BREAKDOWN BY CITY
────────────────────────────────────────────────────────────
  Chicago                64 policies  [2 individual, 13 parent, 49 sub]
    [parent    ] Retrofit buildings to reduce energy use and GHG emissions across residential, multifamily,…
    [sub       ] Retrofit residential buildings with 4 or fewer units: 20% by 2030 and 50% by 2040, priorit…
    [sub       ] Retrofit 20% of total 5+ unit residential buildings by 2030.
    [sub       ] Retrofit 20% of total industrial buildings by 2030.
    [sub       ] Retrofit 90% of total city-owned and sister agency-owned buildings by 2035.
    [sub       ] Retrofit 20% of total commercial buildings by 2035.
    [parent    ] Connect communities to renewable energy through community ownership models and expanded re…
    [sub       ] Install 5 megawatts of co-owned community solar projects by 2025

2026/03/03 20:07:38 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].
2026/03/03 20:07:38 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].
2026/03/03 20:07:38 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].
2026/03/03 20:07:38 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].
2026/03/03 20:07:38 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_

Processed 1 / 121 examples:   0%|          | 0/121 [00:00<?, ?it/s]

2026/03/03 20:07:38 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].


Processed 3 / 121 examples:   2%|▏         | 2/121 [00:00<00:00, 326.15it/s] 

...

Processed 119 / 121 examples:  98%|█████████▊| 118/121 [00:01<00:00, 84.26it/s]

2026/03/03 20:07:40 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].


Processed 120 / 121 examples:  98%|█████████▊| 119/121 [00:01<00:00, 84.26it/s]

2026/03/03 20:07:40 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['canonical_mechanism', 'representative_policies']. Missing: ['sector', 'mechanism_description'].


Processed 121 / 121 examples: 100%|██████████| 121/121 [00:01<00:00, 87.27it/s]

Mechanism registry: 121 unique mechanisms

  building_retrofit → energy_use_reduction
    Primary:   Mitigation
    Secondary: Resource Efficiency
    Policies:  1 across Chicago

  residential_building_retrofit → energy_use_reduction
    Primary:   Mitigation
    Secondary: Resource Efficiency
    Policies:  4 across Austin, Chicago, Miami_Dade

  commercial_building_retrofit → energy_use_reduction
    Primary:   Mitigation
    Secondary: Resource Efficiency
    Policies:  4 across Chicago, Dakar

  renewable_energy_target → grid_decarbonization
    Primary:   Mitigation
    Secondary: None
    Policies:  7 across Chicago, Dakar, Miami_Dade

  solar_pv_deployment → fossil_generation_displacement
    Primary:   Mitigation
    Secondary: Resource Efficiency
    Policies:  5 across Chicago, Miami_Dade

  community_renewables_subscription_expansion → fossil_generation_displacement
    Primary:   Mitigation
  

In [ ]:
# ── Post-Step 8: Enrich combined_policies.csv + write per-policy trace files ──
#
# For every city that has a classified_policies.csv this step:
#   1. Merges a minimal set of classification columns into combined_policies.csv
#      (canonical_mechanism, primary_category, secondary_categories,
#       instrument_type, climate_relevance, mechanism_confidence)
#   2. Writes one JSON file per policy under outputs/<City>/policy_traces/
#      containing the full classification trace for that policy.

import re

_CLASSIFICATION_COLS = [
    "canonical_mechanism",
    "primary_category",
    "secondary_categories",
    "instrument_type",
    "climate_relevance",
    "mechanism_confidence",
]

_TRACE_POLICY_COLS = [
    "policy_statement", "role", "sector",
    "canonical_mechanism", "mechanism_description",
    "primary_category", "secondary_categories", "secondary_justification",
    "primary_causal_pathway", "causal_mechanism_detail", "dominant_pathway_test",
    "mechanism_classification_reasoning", "mechanism_confidence", "mechanism_edge_case_notes",
    "additional_secondary_evidence",
    "instrument_type", "instrument_directness", "climate_relevance",
    "key_indicators", "co_benefits", "instance_edge_case_notes",
]

def _safe_filename(s: str, maxlen: int = 60) -> str:
    s = re.sub(r"[^\w\s-]", "", str(s)).strip()
    s = re.sub(r"[\s-]+", "_", s)
    return s[:maxlen]

for entry in BATCH:
    key = city_key(entry["metadata"])
    city_dir = OUTPUT_DIR / key
    classified_path = city_dir / "classified_policies.csv"
    combined_path   = city_dir / "combined_policies.csv"

    if not classified_path.exists():
        print(f"[{key}] ⚠ classified_policies.csv not found — skipping.")
        continue

    df_cls = pd.read_csv(classified_path)

    # ── 1. Enrich combined_policies.csv ────────────────────────────────────────
    if combined_path.exists():
        df_comb = pd.read_csv(combined_path)
        add_cols = [c for c in _CLASSIFICATION_COLS if c in df_cls.columns and c not in df_comb.columns]
        if add_cols:
            # join on policy_statement (unique enough within a city)
            df_comb = df_comb.merge(
                df_cls[["policy_statement"] + add_cols].drop_duplicates(subset=["policy_statement"]),
                on="policy_statement",
                how="left",
            )
            df_comb.to_csv(combined_path, index=False)
            print(f"[{key}] combined_policies.csv enriched with: {add_cols}")
        else:
            print(f"[{key}] combined_policies.csv already has classification columns — skipped.")
    else:
        print(f"[{key}] ⚠ combined_policies.csv not found — skipping enrichment.")

    # ── 2. Per-policy trace files ──────────────────────────────────────────────
    traces_dir = city_dir / "policy_traces"
    traces_dir.mkdir(parents=True, exist_ok=True)

    trace_cols = [c for c in _TRACE_POLICY_COLS if c in df_cls.columns]
    written = 0
    for idx, row in df_cls.iterrows():
        trace = {c: (None if pd.isna(row[c]) else row[c]) for c in trace_cols}
        stmt_slug = _safe_filename(row.get("policy_statement", f"policy_{idx}"))
        fname = f"{idx:03d}_{stmt_slug}.json"
        with open(traces_dir / fname, "w", encoding="utf-8") as f:
            json.dump(trace, f, ensure_ascii=False, indent=2)
        written += 1

    print(f"[{key}] {written} per-policy traces → {traces_dir}")

print("\nDone.")


[Chicago] combined_policies.csv already has classification columns — skipped.
[Chicago] 64 per-policy traces → outputs/Chicago/policy_traces
[Seattle] combined_policies.csv already has classification columns — skipped.
[Seattle] 5 per-policy traces → outputs/Seattle/policy_traces
[Las_Vegas] combined_policies.csv already has classification columns — skipped.
[Las_Vegas] 13 per-policy traces → outputs/Las_Vegas/policy_traces
[Miami_Dade] combined_policies.csv already has classification columns — skipped.
[Miami_Dade] 28 per-policy traces → outputs/Miami_Dade/policy_traces
[Austin] combined_policies.csv already has classification columns — skipped.
[Austin] 93 per-policy traces → outputs/Austin/policy_traces
[Dakar] combined_policies.csv already has classification columns — skipped.
[Dakar] 3 per-policy traces → outputs/Dakar/policy_traces
[Kuwait] combined_policies.csv already has classification columns — skipped.
[Kuwait] 14 per-policy traces → outputs/Kuwait/policy_traces
[Portugal] c